In [13]:
import pandas as pd

df = pd.read_csv('model1-lama.csv', delimiter=';')

In [14]:
df

,date,usd_idr_close,usd_idr_open,usd_idr_high,usd_idr_low,usd_idr_returns,id_inflation_rate_yoy,id_inflation_rate_mom,bi_interest_rate,idr_m1,idr_m2,id_leading_economic_indicator,total_credit,non_performing_loan,npl_ratio
0,01/01/15,"12,665.00","12,425.00","12,757.50","12,410.00",2.30%,6.96%,-0.24,7.75%,"918,079","4,174,826",99.99,"3,634,620","86,117",2.37%
1,01/02/15,"12,920.00","12,710.00","12,945.00","12,587.50",2.01%,6.29%,-0.36,7.50%,"927,848","4,218,123",99.92,"3,665,686","89,072",2.43%
2,01/03/15,"13,070.00","12,975.00","13,248.00","12,900.00",1.16%,6.38%,0.17,7.50%,"957,580","4,246,361",99.78,"3,679,871","88,401",2.40%
3,01/04/15,"12,960.00","13,070.00","13,077.50","12,805.00",-0.84%,6.79%,0.36,7.50%,"959,376","4,275,711",99.54,"3,711,569","92,142",2.48%
4,01/05/15,"13,223.00","12,985.00","13,243.00","12,976.00",2.03%,7.15%,0.50,7.50%,"980,915","4,288,369",99.25,"3,757,133","97,092",2.58%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,01/09/24,"15,135.00","15,497.50","15,583.50","15,065.00",-2.04%,1.84%,-0.12,6.00%,"2,672,408","9,047,999",100.30,"7,579,250","167,192",2.21%
117,01/10/24,"15,690.00","15,175.00","15,778.50","15,167.50",3.67%,1.71%,0.08,6.00%,"2,697,741","9,082,755",100.37,"7,656,895","168,329",2.20%
118,01/11/24,"15,840.00","15,705.00","15,971.50","15,617.50",0.96%,1.55%,0.30,6.00%,"2,822,294","9,175,751",100.35,"7,717,257","168,963",2.19%
119,01/12/24,"16,090.00","15,870.00","16,322.50","15,830.00",1.58%,1.57%,0.44,6.00%,"2,839,485","9,210,816",100.28,"7,831,558","162,906",2.08%


In [15]:
bi_interest_rate = df[['date', 'bi_interest_rate']].reset_index(drop=True)

In [16]:
bi_interest_rate

,date,bi_interest_rate
0,01/01/15,7.75%
1,01/02/15,7.50%
2,01/03/15,7.50%
3,01/04/15,7.50%
4,01/05/15,7.50%
...,...,...
116,01/09/24,6.00%
117,01/10/24,6.00%
118,01/11/24,6.00%
119,01/12/24,6.00%


In [17]:
bi_interest_rate.to_csv('bi_interest_rate.csv', index=False)

In [18]:
npl = df[['date', 'non_performing_loan']].reset_index(drop=True)

In [19]:
npl.to_csv('npl.csv', index=False)

In [20]:
nplratio = df[['date', 'npl_ratio']].reset_index(drop=True)

In [21]:
nplratio.to_csv('npl_ratio.csv', index=False)

In [22]:
import pandas as pd
import numpy as np
from datetime import datetime

def clean_cpi_dataset(df_cpi):
    """
    Clean CPI dataset by removing yearly/quarterly rows and fixing date format.
    
    Parameters:
    -----------
    df_cpi : pandas.DataFrame
        Raw CPI data with mixed yearly, quarterly, and monthly entries
    
    Returns:
    --------
    pandas.DataFrame
        Cleaned DataFrame with only monthly data and proper date format
    """
    
    # Make a copy to avoid modifying original
    df_clean = df_cpi.copy()
    
    # Assuming columns are 'Date' and 'CPI' (adjust if different)
    if df_clean.columns.tolist() == [0, 1]:
        df_clean.columns = ['Date', 'CPI']
    elif 'Unnamed: 0' in df_clean.columns:
        df_clean.columns = ['Date', 'CPI']
    
    print("Original dataset shape:", df_clean.shape)
    print("First 10 rows:")
    print(df_clean.head(10))
    print("\nUnique date patterns:")
    print(df_clean['Date'].str.extract(r'(\d{4}(?:-[MQ]\d{2})?(?:-\w+)?)')[0].value_counts().head(10))
    
    # Step 1: Remove rows with yearly data only (e.g., "1983", "1984")
    yearly_pattern = r'^\d{4}$'
    yearly_mask = df_clean['Date'].str.match(yearly_pattern, na=False)
    print(f"\nRemoving {yearly_mask.sum()} yearly rows...")
    
    # Step 2: Remove rows with quarterly data (e.g., "1983-Q1", "1984-Q2")  
    quarterly_pattern = r'^\d{4}-Q[1-4]$'
    quarterly_mask = df_clean['Date'].str.match(quarterly_pattern, na=False)
    print(f"Removing {quarterly_mask.sum()} quarterly rows...")
    
    # Step 3: Keep only monthly data (e.g., "1983-M01", "1984-M12")
    monthly_pattern = r'^\d{4}-M\d{2}$'
    monthly_mask = df_clean['Date'].str.match(monthly_pattern, na=False)
    print(f"Keeping {monthly_mask.sum()} monthly rows...")
    
    # Filter to keep only monthly data
    df_monthly = df_clean[monthly_mask].copy()
    
    # Step 4: Convert date format from "1983-M01" to proper datetime
    def convert_date_format(date_str):
        """Convert '1983-M01' to datetime object"""
        try:
            # Extract year and month from "YYYY-MXX" format
            year = int(date_str[:4])
            month = int(date_str[6:8])  # Skip "M" and get the two digits
            
            # Create datetime object (use day 1 for consistency)
            return datetime(year, month, 1)
        except:
            return pd.NaT
    
    print("\nConverting date format...")
    df_monthly['Date'] = df_monthly['Date'].apply(convert_date_format)
    
    # Step 5: Remove any rows with invalid dates or missing CPI values
    initial_count = len(df_monthly)
    df_monthly = df_monthly.dropna(subset=['Date', 'CPI']).reset_index(drop=True)
    dropped_count = initial_count - len(df_monthly)
    if dropped_count > 0:
        print(f"Removed {dropped_count} rows with missing data...")
    
    # Step 6: Convert CPI to numeric (handle any string formatting issues)
    df_monthly['CPI'] = pd.to_numeric(df_monthly['CPI'], errors='coerce')
    
    # Step 7: Sort by date and reset index
    df_monthly = df_monthly.sort_values('Date').reset_index(drop=True)
    
    # Step 8: Create Month-End frequency for consistency with Model 1
    df_monthly['Date'] = pd.to_datetime(df_monthly['Date']) + pd.offsets.MonthEnd(0)
    
    print(f"\n✅ Cleaning completed!")
    print(f"Final dataset shape: {df_monthly.shape}")
    print(f"Date range: {df_monthly['Date'].min()} to {df_monthly['Date'].max()}")
    print(f"Total months: {len(df_monthly)}")
    
    return df_monthly

# Step-by-step cleaning process
def clean_cpi_step_by_step(file_path):
    """
    Complete CPI cleaning workflow with diagnostics
    """
    
    # Load the raw data
    print("Loading CPI dataset...")
    try:
        df_raw = pd.read_csv(file_path, header=None, names=['Date', 'CPI'])
    except:
        # If it has headers or different format
        df_raw = pd.read_csv(file_path)
        if len(df_raw.columns) == 2:
            df_raw.columns = ['Date', 'CPI']
    
    print("Raw data sample:")
    print(df_raw.head(15))
    
    # Clean the dataset
    df_clean = clean_cpi_dataset(df_raw)
    
    # Display results
    print("\n" + "="*60)
    print("CLEANED CPI DATASET")
    print("="*60)
    print(df_clean.head(10))
    print("\nDataset info:")
    print(df_clean.info())
    print(f"\nCPI Statistics:")
    print(df_clean['CPI'].describe())
    
    # Check for any gaps in monthly data
    print(f"\nChecking for missing months...")
    date_range = pd.date_range(start=df_clean['Date'].min(), 
                              end=df_clean['Date'].max(), 
                              freq='ME')  # Month-End frequency
    
    missing_dates = set(date_range) - set(df_clean['Date'])
    if missing_dates:
        print(f"⚠️ Found {len(missing_dates)} missing months:")
        for date in sorted(missing_dates)[:10]:  # Show first 10
            print(f"   {date.strftime('%Y-%m')}")
        if len(missing_dates) > 10:
            print(f"   ... and {len(missing_dates) - 10} more")
    else:
        print("✅ No missing months detected")
    
    return df_clean

# Usage example:
if __name__ == "__main__":
    # Clean your CPI dataset
    cpi_cleaned = clean_cpi_step_by_step('cpi.csv')  # Adjust filename as needed
    
    # Save cleaned version
    cpi_cleaned.to_csv('dataset/cpi_cleaned.csv', index=False)
    print(f"\n💾 Cleaned dataset saved to: dataset/cpi_cleaned.csv")
    
    # Show recent data
    print(f"\nMost recent CPI data:")
    print(cpi_cleaned.tail(10))
    
    # Show early data  
    print(f"\nEarliest CPI data:")
    print(cpi_cleaned.head(10))

Loading CPI dataset...
Raw data sample:
        Date       CPI
0   1968-M01  0.456958
1   1968-M02  0.502654
2   1968-M03  0.517886
3   1968-M04  0.487422
4   1968-M05  0.512809
5   1968-M06  0.528041
6   1968-M07  0.553427
7   1968-M08  0.568659
8   1968-M09  0.568659
9   1968-M10  0.563582
10  1968-M11  0.690515
11  1968-M12  0.609278
12  1969-M01  0.619432
13  1969-M02  0.629587
14  1969-M03  0.644819
Original dataset shape: (915, 2)
First 10 rows:
       Date       CPI
0  1968-M01  0.456958
1  1968-M02  0.502654
2  1968-M03  0.517886
3  1968-M04  0.487422
4  1968-M05  0.512809
5  1968-M06  0.528041
6  1968-M07  0.553427
7  1968-M08  0.568659
8  1968-M09  0.568659
9  1968-M10  0.563582

Unique date patterns:
0
1968-M01    1
1968-M02    1
1968-M03    1
1968-M04    1
1968-M05    1
1968-M06    1
1968-M07    1
1968-M08    1
1968-M09    1
1968-M10    1
Name: count, dtype: int64

Removing 43 yearly rows...
Removing 175 quarterly rows...
Keeping 697 monthly rows...

Converting date format.

In [ ]:
# Quick cleaning function
def quick_clean_cpi(df):
    """
    Quick CPI dataset cleaning - remove yearly/quarterly rows, fix dates
    """
    # Keep only monthly data (pattern: YYYY-MXX)
    monthly_mask = df.iloc[:, 0].str.match(r'^\d{4}-M\d{2}$', na=False)
    df_monthly = df[monthly_mask].copy()
    
    # Convert date format: "1983-M01" → datetime
    df_monthly.iloc[:, 0] = pd.to_datetime(
        df_monthly.iloc[:, 0].str.replace('M', ''), 
        format='%Y-%m'
    ) + pd.offsets.MonthEnd(0)  # Convert to month-end for consistency
    
    # Clean numeric data
    df_monthly.iloc[:, 1] = pd.to_numeric(df_monthly.iloc[:, 1], errors='coerce')
    
    # Set proper column names and sort
    df_monthly.columns = ['Date', 'CPI']
    df_monthly = df_monthly.sort_values('Date').reset_index(drop=True)
    
    return df_monthly

# Usage:
# cpi_cleaned = quick_clean_cpi(cpi_df)

In [24]:
usd_idr = pd.read_csv('usd-idr.csv')

In [25]:
usd_idr

,date,Terakhir
0,23/02/2026,"16.853,3"
1,22/02/2026,"16.853,3"
2,20/02/2026,"16.865,0"
3,19/02/2026,"16.875,0"
4,18/02/2026,"16.880,0"
...,...,...
2849,7/1/2015,"12.738,5"
2850,6/1/2015,"12.657,5"
2851,5/1/2015,"12.627,5"
2852,2/1/2015,"12.542,5"
